# Prompt sensitivity

**Session 3 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

Before you believe a "3-point improvement" from a reword, measure two things: the **noise
floor** (same prompt, run many times) and the **variant spread** (trivial rewordings). On
this task the two are about equal — so most single-run prompt A/Bs here are measuring
nothing. The wording change that *does* move the number is the one that changes the label
vocabulary, and that is a scoring artefact, not a better prompt.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from eval import load_cases, run_eval, print_report, exact
from utils import ask, SMALL_MODEL, BIG_MODEL


### 1. The noise floor

Run the exact same prompt 10 times on the ticket set. The spread of those 10 accuracies is
the smallest difference you could ever attribute to a prompt change.

In [ ]:
cases = load_cases("../eval/datasets/support_tickets.jsonl")
LABELS = ["billing", "bug", "other"]

BASE = (
    'Classify the support ticket as billing, bug, or other.\n'
    'Reply with ONE lowercase word.\n\n'
    'Ticket: "I cannot log in since the update" -> bug\n'
    'Ticket: "Refund me for the double charge" -> billing\n'
    'Ticket: "What are your office hours?" -> other\n\n'
    'Ticket: "{t}" ->'
)

def clf(prompt_tmpl, model=SMALL_MODEL):
    def f(text):
        out = ask(prompt_tmpl.format(t=text), model=model).strip().lower()
        return next((lab for lab in LABELS if lab in out), out)
    return f

floor = run_eval(cases, clf(BASE), scorer=exact, repeats=10)
print("same prompt x10:", [f"{a:.0%}" for a in floor["accuracies"]])
print(f"noise floor (spread): {floor['spread']:.0%}")


### 2. Trivial rewordings vs one real change

Five variants that should not matter (reordered examples, colon instead of arrow, a polite
preamble, lowercase) and one that does: renaming the label `bug` to `issue` in the prompt
while the eval set still expects `bug`.

In [ ]:
variants = {
    "baseline":          BASE,
    "examples reordered": ('Classify the support ticket as billing, bug, or other.\n'
                           'Reply with ONE lowercase word.\n\n'
                           'Ticket: "What are your office hours?" -> other\n'
                           'Ticket: "Refund me for the double charge" -> billing\n'
                           'Ticket: "I cannot log in since the update" -> bug\n\n'
                           'Ticket: "{t}" ->'),
    "colon not arrow":   BASE.replace(" -> ", ": "),
    "polite preamble":   "Hi, could you help me sort this ticket please?\n\n" + BASE,
    "lowercased":        BASE.lower(),
    "label bug->issue":  BASE.replace("bug", "issue"),   # the one that actually moves it
}

results = {name: run_eval(cases, clf(v), scorer=exact, repeats=3) for name, v in variants.items()}
for name, r in results.items():
    print(f"  {name:20} {r['acc_mean']:.0%}  (spread {r['spread']:.0%})")

trivial = [results[n]["acc_mean"] for n in list(variants)[:5]]
print(f"\n  trivial-variant spread: {max(trivial) - min(trivial):.0%}   noise floor: {floor['spread']:.0%}")
print(f"  label bug->issue:       {results['label bug->issue']['acc_mean']:.0%}  <- not a worse prompt, a broken scorer")


## Your turn - vary the example

1. Add two more trivial variants (extra whitespace, punctuation, a "Please").
2. Which variant wins? Re-run once more - is the winner stable or noise?
3. A prompt that only wins by luck is not done. What spread would you accept?
